# 日内早盘动量稳定超额研究

只用 qlib 原生 15 分钟 provider，检验开盘 60 分钟可观测路径能否映射为 10:30–收盘的
SOXL/SOXX/现金仓位并稳定超越 SOXX。无收益预测、无拟合参数、无隔夜仓。


## 1. 预登记计划


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'intraday_early_momentum_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('provider:', plan['provider_uri'], '| as_of:', plan['as_of'])
print('segments:', plan['segments'])


plan sha256: ff271e5c8a33dac9b1a605cd3d781c9e9be3aae6a3e27ee9bd1c1d499e231c4b
provider: data/qlib/soxl_intraday_15min | as_of: 2026-10-07
segments: {'development': ['2020-01-02', '2022-12-30'], 'reused_diagnostic': ['2023-01-03', '2026-09-25'], 'forward': ['2026-09-26', '2026-10-07']}


## 2. 运行（qlib 原生 + Recorder）


In [2]:
from src.workflow import intraday_early_momentum as workflow
from src.workflow.intraday_early_momentum import write_card_files

summary = workflow.run_intraday_early_momentum(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('selected policy:', summary['selected_policy'])
print('promotion_eligible:', summary['promotion_eligible'])
print('recorder_id:', summary['recorder_id'])
print('provider_uri:', summary['provider_uri'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[37108:MainThread](2026-10-09 16:17:35,614) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/09 16:17:35 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[37108:MainThread](2026-10-09 16:17:35,989) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[37108:MainThread](2026-10-09 16:17:35,990) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


...

[37108:MainThread](2026-10-09 16:17:36,593) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name intraday_early_momentum_20261010.


[37108:MainThread](2026-10-09 16:17:36,594) INFO - qlib.workflow - [exp.py:258] - Experiment 381826478494092836 starts running ...


[37108:MainThread](2026-10-09 16:17:36,646) INFO - qlib.workflow - [recorder.py:345] - Recorder 90422ab9e8504b8b864d0d67c957f448 starts running under Experiment 381826478494092836 ...


[37108:MainThread](2026-10-09 16:17:36,721) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


decision: no_stable_intraday_excess_in_registered_family
selected policy: early_momentum_l2
promotion_eligible: False
recorder_id: 90422ab9e8504b8b864d0d67c957f448
provider_uri: /Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min


## 3. 候选对比（10bp）


In [3]:
import pandas as pd
comp = pd.read_csv(CARD / 'policy_comparison.csv')
cols = ['window','policy','return_pct','benchmark_return_pct','max_drawdown_pct',
        'benchmark_max_drawdown_pct','sharpe','benchmark_sharpe','sharpe_difference',
        'stable_excess','positive_year_count','year_count','total_turnover']
display(comp[cols].round(3))


,window,policy,return_pct,benchmark_return_pct,max_drawdown_pct,benchmark_max_drawdown_pct,sharpe,benchmark_sharpe,sharpe_difference,stable_excess,positive_year_count,year_count,total_turnover
0,development,early_momentum_off,-11.932,-11.932,-32.644,-32.644,-0.049,-0.049,0.000,False,0,3,0.0
1,development,early_momentum_l1,-52.335,-11.932,-68.354,-32.644,-0.250,-0.049,-0.201,False,1,3,680.0
2,development,early_momentum_l2,-13.943,-11.932,-50.396,-32.644,0.106,-0.049,0.155,False,2,3,556.0
3,development,early_momentum_l3,-26.667,-11.932,-53.390,-32.644,-0.040,-0.049,0.009,False,2,3,424.0
4,development,early_momentum_l5,-34.177,-11.932,-47.445,-32.644,-0.235,-0.049,-0.185,False,0,3,136.0
5,development,early_momentum_l2_s2,-40.523,-11.932,-59.844,-32.644,-0.205,-0.049,-0.156,False,2,3,906.0
6,development,early_momentum_l3_s3,-46.134,-11.932,-63.169,-32.644,-0.315,-0.049,-0.266,False,2,3,690.0
7,reused_diagnostic,early_momentum_off,47.066,47.066,-35.117,-35.117,0.580,0.580,0.000,False,0,4,0.0
8,reused_diagnostic,early_momentum_l1,25.043,47.066,-59.772,-35.117,0.348,0.580,-0.232,False,1,4,880.0
9,reused_diagnostic,early_momentum_l2,27.922,47.066,-61.368,-35.117,0.357,0.580,-0.223,False,2,4,680.0


## 4. 选中策略年度分解


In [4]:
display(pd.read_csv(CARD / 'annual_metrics.csv').round(4))
print('forward verdict:', json.dumps(summary['selected_forward_verdict'], indent=2))


,window,policy,year,sessions,wealth,average_soxl_weight,average_cash_weight,turnover,sessions_bench,wealth_bench,average_soxl_weight_bench,average_cash_weight_bench,turnover_bench,wealth_ratio_vs_soxx
0,development,early_momentum_l2,2020,251,0.6404,0.2550,0.0,184.0,251,0.8296,0.0,0.0,0.0,0.7719
1,development,early_momentum_l2,2021,251,1.2260,0.2271,0.0,164.0,251,1.1237,0.0,0.0,0.0,1.0910
2,development,early_momentum_l2,2022,250,1.0960,0.3080,0.0,208.0,250,0.9446,0.0,0.0,0.0,1.1602
3,reused_diagnostic,early_momentum_l2,2023,248,1.5470,0.1976,0.0,160.0,248,1.4371,0.0,0.0,0.0,1.0765
4,reused_diagnostic,early_momentum_l2,2024,249,0.7177,0.2169,0.0,168.0,249,0.8511,0.0,0.0,0.0,0.8433
5,reused_diagnostic,early_momentum_l2,2025,247,0.7933,0.2389,0.0,188.0,247,0.9596,0.0,0.0,0.0,0.8267
6,reused_diagnostic,early_momentum_l2,2026,182,1.4522,0.3132,0.0,164.0,182,1.2530,0.0,0.0,0.0,1.1590
7,forward,early_momentum_l2,2026,8,0.9869,0.2500,0.0,6.0,8,1.0041,0.0,0.0,0.0,0.9829


forward verdict: {
  "beats_return_and_drawdown": false,
  "beats_sharpe": false,
  "positive_year_count": 0,
  "year_count": 1,
  "minimum_positive_years": 2,
  "annual_win_rate": 0.0,
  "stable_excess": false,
  "reference": "SOXX buy-and-hold, identical sessions and cost model",
  "scope": "reused_history_and_forward_diagnostic_not_production_evidence"
}


## 5. 中文结论


In [5]:
o = summary['selected_outer']['strategy']; f = summary['selected_forward']['strategy']
print(f"选中 {summary['selected_policy']}；候选族 {summary['candidate_count']} 条；无收益预测、无隔夜仓。")
print(f"复用区间 {summary['segments']['reused_diagnostic']}：收益 {o['return_pct']:.2f}%，回撤 {o['max_drawdown_pct']:.2f}%，Sharpe {o['sharpe']:.2f}。")
print(f"前向区间 {summary['segments']['forward']}：收益 {f['return_pct']:.2f}%，回撤 {f['max_drawdown_pct']:.2f}%，Sharpe {f['sharpe']:.2f}。")
print('是否晋升:', summary['promotion_eligible'], '| 决定:', summary['decision'])
print('qlib Recorder:', summary['recorder_id'])


选中 early_momentum_l2；候选族 7 条；无收益预测、无隔夜仓。
复用区间 ['2023-01-03', '2026-09-25']：收益 27.92%，回撤 -61.37%，Sharpe 0.36。
前向区间 ['2026-09-26', '2026-10-07']：收益 -1.31%，回撤 -3.71%，Sharpe -1.57。
是否晋升: False | 决定: no_stable_intraday_excess_in_registered_family
qlib Recorder: 90422ab9e8504b8b864d0d67c957f448
